In [ ]:
import os
os.environ["FASTFUELS_API_KEY"] = "api-key"
from fastfuels_sdk.domains import Domain
from fastfuels_sdk.features import Features
from fastfuels_sdk.grids import TopographyGridBuilder
from fastfuels_sdk.inventories import Inventories
from fastfuels_sdk.grids import TreeGridBuilder
from fastfuels_sdk.grids import SurfaceGridBuilder
from fastfuels_sdk.grids import Grids
from fastfuels_sdk.pointclouds import PointClouds

import geopandas as gpd
from pathlib import Path


from rasterio.transform import Affine
import rasterio
from rasterio import features
import pandas as pd

In [ ]:
def create_fastfuels(boundary, site_name, product, 
                    version, interpolation_method, 
                    curing_live_herbaceous, curing_live_woody, 
                    groups, feature_masks, remove_non_burnable, 
                    uniform_fuel_moisture_value, grid_data_zip_path, live_fuel_moisture,
                    tree_inventory_save_path, grid_data_qf_path,
                    meta = False, als = False, tree_inventory = False, tree_inventory_path = None,
                    horizontal_resolution=2.0, vertical_resolution=1.0):
    # Create domain
    domain = Domain.from_geodataframe(
    geodataframe=boundary,
    name=site_name,
    description=f"domain of {site_name}",
    horizontal_resolution=horizontal_resolution,  # Use the argument
    vertical_resolution=vertical_resolution       # Use the argument
    )
    print(f"Created domain with ID: {domain.id}")
    
    
    # Initialize Features for our domain
    features = Features.from_domain_id(domain.id)
    
    # Initialize grids for our domain
    grids = Grids.from_domain_id(domain.id)
    
    # Create features from OpenStreetMap
    road_feature = features.create_road_feature_from_osm()
    water_feature = features.create_water_feature_from_osm()
    
    # Wait for features to be ready
    road_feature.wait_until_completed()
    water_feature.wait_until_completed()
    
    print("created road and water")
    
    # Create a feature grid with roads and water bodies
    feature_grid = grids.create_feature_grid(
        attributes=["road", "water"]
    )
    feature_grid.wait_until_completed()
    print("created feature grid")
    
    topography_grid = (
        TopographyGridBuilder(domain_id=domain.id)
        .with_elevation_from_3dep()
        .with_slope_from_3dep()
        .with_aspect_from_3dep()
        .build()
    )
    
    topography_grid.wait_until_completed()
    print("created topo")
    
    # if tree meta flag is true (you want to use a meta as the tree inventory)
    if meta:
        tree_inventory = Inventories.from_domain_id(
            domain.id
        ).create_tree_inventory_from_treemap(
            feature_masks=feature_masks,
            canopy_height_map_source = 'Meta2024'
        )
        tree_inventory.wait_until_completed(step = 1, verbose = True)
        print("created tree")
    elif als:
        als_pointcloud = (
            PointClouds.from_domain_id(domain.id)
            .create_als_point_cloud(sources=["3DEP"])
        )
        als_pointcloud.wait_until_completed(step=1, verbose=True)
        print("created als point cloud")

        tree_inventory = Inventories.from_domain_id(
            domain.id
        ).create_tree_inventory_from_point_cloud()
        tree_inventory.wait_until_completed(step = 1, verbose = True)
    
    
    # else Create tree inventory
    else:
        if use_tree_inventory:
            tree_inventory = Inventories.from_domain_id(
                        domain.id
                    ).create_tree_inventory_from_file_upload(
                        # feature_masks=feature_masks,
                        file_path=(tree_inventory_path)
                    )
            tree_inventory.wait_until_completed(step = 1, verbose = True)
            print("created tree using csv tree inventory")
            
        else:
            tree_inventory = Inventories.from_domain_id(
                domain.id
            ).create_tree_inventory_from_treemap(
                feature_masks=feature_masks
            )
            tree_inventory.wait_until_completed(step = 1, verbose = True)
            print("created tree")

    #export tree inventory
    export = tree_inventory.create_export("csv")  # or "parquet" or "geojson"
    export = export.wait_until_completed()
    
    # Download to specific file
    export.to_file(tree_inventory_save_path)
    
    # Create tree grid
    tree_grid = (
        TreeGridBuilder(domain_id=domain.id)
        .with_bulk_density_from_tree_inventory()
        .with_spcd_from_tree_inventory()
        .with_uniform_fuel_moisture(live_fuel_moisture)
        .build()
    )
    tree_grid.wait_until_completed(step = 1, verbose = True)
    
    print("created treegrid")
    
    # Create surface grid
    if product == "FBFM40":
        surface_grid = (
            SurfaceGridBuilder(domain_id=domain.id)
            .with_fuel_load_from_landfire(
                product=product,
                version=version,
                interpolation_method=interpolation_method,
                curing_live_herbaceous=curing_live_herbaceous,
                curing_live_woody=curing_live_woody,
                groups=groups,
                feature_masks=feature_masks,
                remove_non_burnable=remove_non_burnable,
            )
            .with_fuel_depth_from_landfire(
                product=product,
                version=version,
                interpolation_method=interpolation_method,
                feature_masks=feature_masks,
                remove_non_burnable=remove_non_burnable,
            )
            .with_uniform_fuel_moisture(
                value=uniform_fuel_moisture_value,
                feature_masks=feature_masks
            )
            .build()
        )
    elif product == "FCCS":
        surface_grid = (
            SurfaceGridBuilder(domain_id=domain.id)
            .with_fuel_load_from_landfire(
                product=product,
                version=version,
                interpolation_method=interpolation_method,
                groups=groups,
                feature_masks=feature_masks,
            )
            .with_fuel_depth_from_landfire(
                product=product,
                version=version,
                interpolation_method=interpolation_method,
                feature_masks=feature_masks,
            )
            .with_uniform_fuel_moisture(
                value=uniform_fuel_moisture_value,
                feature_masks=feature_masks
            )
            .build()
        )
    surface_grid.wait_until_completed(step = 1, verbose = True)
    print("created surface")
    
    # Create the export
    export = grids.create_export("zarr") #"zarr" or "QUICFire"
    
    # Wait for the export to complete
    export.wait_until_completed()
    print('exported to zarr')
    
    # Save to a file
    export.to_file(grid_data_zip_path)

    export = grids.create_export("QUIC-Fire") #"zarr" or "QUIC-Fire"
    
    # Wait for the export to complete
    export.wait_until_completed()
    print('exported to QUIC-Fire')
    # Save to a file
    export.to_file(grid_data_qf_path)

In [ ]:
# set the parameters to create the fastfuels tile

project_folder = "/Users/leticia/Documents/Data/plum_ranch/"#sys.argv[2]#"/Users/leticia/Documents/Data/independence_lake/"
gui_folder = "/Users/leticia/Documents/OCT_2025/Mac_GUI/projects/plum_ranch/"#sys.argv[3]#"/Users/leticia/Documents/Mac_GUI_sept/GUI/projects/als_il2/"



boundary = gpd.read_file(gui_folder + 'ff_domain.geojson')
boundary = boundary.to_crs(5070)
site_name = 'sitename'
product="FBFM40"
version='2022'
interpolation_method='zipper'
curing_live_herbaceous=1
curing_live_woody=0.5
groups = ['oneHour']
feature_masks=["road", "water"]
remove_non_burnable=["NB1", "NB2"]
uniform_fuel_moisture_value=15
live_fuel_moisture=106
use_tree_inventory = False
tree_inventory_path = None
horizontal_resolution=2.0
vertical_resolution=1.0


In [ ]:
grid_data_zip_path = project_folder + 'data/fastfuels/new_trees_2013.zarr.zip'
grid_data_qf_path = project_folder + 'data/fastfuels/new_trees_2013.qf.zip'

tree_inventory_path = project_folder + 'data/treelist_2013.csv'
tree_inventory_save_path = project_folder + 'data/fastfuels/tree_inventory_1_export.csv'


create_fastfuels(boundary, site_name, product, 
                    version, interpolation_method, 
                    curing_live_herbaceous, curing_live_woody, 
                    groups, feature_masks, remove_non_burnable, 
                    uniform_fuel_moisture_value, grid_data_zip_path, live_fuel_moisture,
                    tree_inventory_save_path, grid_data_qf_path,
                    meta = False, als = False, tree_inventory = True, tree_inventory_path = tree_inventory_path,
                    horizontal_resolution=2.0, vertical_resolution=1.0)

In [ ]:
grid_data_zip_path = project_folder + 'data/fastfuels/trees_2013_1.zarr.zip'
grid_data_qf_path = project_folder + 'data/fastfuels/trees_2013_1.qf.zip'

tree_inventory_path = project_folder + 'data/treelist_2013_1.csv'
tree_inventory_save_path = project_folder + 'data/fastfuels/tree_inventory_1_export.csv'
live_fuel_moisture = 1

create_fastfuels(boundary, site_name, product, 
                    version, interpolation_method, 
                    curing_live_herbaceous, curing_live_woody, 
                    groups, feature_masks, remove_non_burnable, 
                    uniform_fuel_moisture_value, grid_data_zip_path, live_fuel_moisture,
                    tree_inventory_save_path, grid_data_qf_path,
                    meta = False, als = False, tree_inventory = True, tree_inventory_path = tree_inventory_path,
                    horizontal_resolution=2.0, vertical_resolution=1.0)



In [ ]:
grid_data_zip_path = project_folder + 'data/fastfuels/trees_2013_2.zarr.zip'
grid_data_qf_path = project_folder + 'data/fastfuels/trees_2013_2.qf.zip'

tree_inventory_path = project_folder + 'data/treelist_2013_2.csv'
tree_inventory_save_path = project_folder + 'data/fastfuels/tree_inventory_1_export.csv'


create_fastfuels(boundary, site_name, product, 
                    version, interpolation_method, 
                    curing_live_herbaceous, curing_live_woody, 
                    groups, feature_masks, remove_non_burnable, 
                    uniform_fuel_moisture_value, grid_data_zip_path, live_fuel_moisture,
                    tree_inventory_save_path, grid_data_qf_path,
                    meta = False, als = False, tree_inventory = True, tree_inventory_path = tree_inventory_path,
                    horizontal_resolution=2.0, vertical_resolution=1.0)

In [ ]:
grid_data_zip_path = project_folder + 'data/fastfuels/trees_2013_3.zarr.zip'
grid_data_qf_path = project_folder + 'data/fastfuels/trees_2013_3.qf.zip'

tree_inventory_path = project_folder + 'data/treelist_2013_3.csv'
tree_inventory_save_path = project_folder + 'data/fastfuels/tree_inventory_1_export.csv'


create_fastfuels(boundary, site_name, product, 
                    version, interpolation_method, 
                    curing_live_herbaceous, curing_live_woody, 
                    groups, feature_masks, remove_non_burnable, 
                    uniform_fuel_moisture_value, grid_data_zip_path, live_fuel_moisture,
                    tree_inventory_save_path, grid_data_qf_path,
                    meta = False, als = False, tree_inventory = True, tree_inventory_path = tree_inventory_path,
                    horizontal_resolution=2.0, vertical_resolution=1.0)

In [ ]:
grid_data_zip_path = project_folder + 'data/fastfuels/trees_2022.zarr.zip'
grid_data_qf_path = project_folder + 'data/fastfuels/trees_2022.qf.zip'

tree_inventory_path = project_folder + 'data/treelist_2022.csv'
tree_inventory_save_path = project_folder + 'data/fastfuels/tree_inventory_1_export.csv'


create_fastfuels(boundary, site_name, product, 
                    version, interpolation_method, 
                    curing_live_herbaceous, curing_live_woody, 
                    groups, feature_masks, remove_non_burnable, 
                    uniform_fuel_moisture_value, grid_data_zip_path, live_fuel_moisture,
                    tree_inventory_save_path, grid_data_qf_path,
                    meta = False, als = False, tree_inventory = True, tree_inventory_path = tree_inventory_path,
                    horizontal_resolution=2.0, vertical_resolution=1.0)

In [ ]:
grid_data_zip_path = project_folder + 'data/fastfuels/trees_2022_1.zarr.zip'
grid_data_qf_path = project_folder + 'data/fastfuels/trees_2022_1.qf.zip'

tree_inventory_path = project_folder + 'data/treelist_2022_1.csv'
tree_inventory_save_path = project_folder + 'data/fastfuels/tree_inventory_1_export.csv'


create_fastfuels(boundary, site_name, product, 
                    version, interpolation_method, 
                    curing_live_herbaceous, curing_live_woody, 
                    groups, feature_masks, remove_non_burnable, 
                    uniform_fuel_moisture_value, grid_data_zip_path, live_fuel_moisture,
                    tree_inventory_save_path, grid_data_qf_path,
                    meta = False, als = False, tree_inventory = True, tree_inventory_path = tree_inventory_path,
                    horizontal_resolution=2.0, vertical_resolution=1.0)

In [ ]:
grid_data_zip_path = project_folder + 'data/fastfuels/trees_2022_2.zarr.zip'
grid_data_qf_path = project_folder + 'data/fastfuels/trees_2022_2.qf.zip'

tree_inventory_path = project_folder + 'data/treelist_2022_2.csv'
tree_inventory_save_path = project_folder + 'data/fastfuels/tree_inventory_1_export.csv'


create_fastfuels(boundary, site_name, product, 
                    version, interpolation_method, 
                    curing_live_herbaceous, curing_live_woody, 
                    groups, feature_masks, remove_non_burnable, 
                    uniform_fuel_moisture_value, grid_data_zip_path, live_fuel_moisture,
                    tree_inventory_save_path, grid_data_qf_path,
                    meta = False, als = False, tree_inventory = True, tree_inventory_path = tree_inventory_path,
                    horizontal_resolution=2.0, vertical_resolution=1.0)

In [ ]:
grid_data_zip_path = project_folder + 'data/fastfuels/trees_2022_3.zarr.zip'
grid_data_qf_path = project_folder + 'data/fastfuels/trees_2022_3.qf.zip'

tree_inventory_path = project_folder + 'data/treelist_2022_3.csv'
tree_inventory_save_path = project_folder + 'data/fastfuels/tree_inventory_1_export.csv'


create_fastfuels(boundary, site_name, product, 
                    version, interpolation_method, 
                    curing_live_herbaceous, curing_live_woody, 
                    groups, feature_masks, remove_non_burnable, 
                    uniform_fuel_moisture_value, grid_data_zip_path, live_fuel_moisture,
                    tree_inventory_save_path, grid_data_qf_path,
                    meta = False, als = False, tree_inventory = True, tree_inventory_path = tree_inventory_path,
                    horizontal_resolution=2.0, vertical_resolution=1.0)

In [ ]:
grid_data_zip_path = project_folder + 'data/fastfuels/trees_2022a.zarr.zip'
grid_data_qf_path = project_folder + 'data/fastfuels/trees_2022a.qf.zip'

tree_inventory_path = project_folder + 'data/treelist_2022a.csv'
tree_inventory_save_path = project_folder + 'data/fastfuels/tree_inventory_1_export.csv'


create_fastfuels(boundary, site_name, product, 
                    version, interpolation_method, 
                    curing_live_herbaceous, curing_live_woody, 
                    groups, feature_masks, remove_non_burnable, 
                    uniform_fuel_moisture_value, grid_data_zip_path, live_fuel_moisture,
                    tree_inventory_save_path, grid_data_qf_path,
                    meta = False, als = False, tree_inventory = True, tree_inventory_path = tree_inventory_path,
                    horizontal_resolution=2.0, vertical_resolution=1.0)

In [ ]:
grid_data_zip_path = project_folder + 'data/fastfuels/trees_2022a_1.zarr.zip'
grid_data_qf_path = project_folder + 'data/fastfuels/trees_2022a_1.qf.zip'

tree_inventory_path = project_folder + 'data/treelist_2022a_1.csv'
tree_inventory_save_path = project_folder + 'data/fastfuels/tree_inventory_1_export.csv'


create_fastfuels(boundary, site_name, product, 
                    version, interpolation_method, 
                    curing_live_herbaceous, curing_live_woody, 
                    groups, feature_masks, remove_non_burnable, 
                    uniform_fuel_moisture_value, grid_data_zip_path, live_fuel_moisture,
                    tree_inventory_save_path, grid_data_qf_path,
                    meta = False, als = False, tree_inventory = True, tree_inventory_path = tree_inventory_path,
                    horizontal_resolution=2.0, vertical_resolution=1.0)

In [ ]:
grid_data_zip_path = project_folder + 'data/fastfuels/trees_2022a_2.zarr.zip'
grid_data_qf_path = project_folder + 'data/fastfuels/trees_2022a_2.qf.zip'

tree_inventory_path = project_folder + 'data/treelist_2022a_2.csv'
tree_inventory_save_path = project_folder + 'data/fastfuels/tree_inventory_1_export.csv'


create_fastfuels(boundary, site_name, product, 
                    version, interpolation_method, 
                    curing_live_herbaceous, curing_live_woody, 
                    groups, feature_masks, remove_non_burnable, 
                    uniform_fuel_moisture_value, grid_data_zip_path, live_fuel_moisture,
                    tree_inventory_save_path, grid_data_qf_path,
                    meta = False, als = False, tree_inventory = True, tree_inventory_path = tree_inventory_path,
                    horizontal_resolution=2.0, vertical_resolution=1.0)

In [ ]:
grid_data_zip_path = project_folder + 'data/fastfuels/trees_2022a_3.zarr.zip'
grid_data_qf_path = project_folder + 'data/fastfuels/trees_2022a_3.qf.zip'

tree_inventory_path = project_folder + 'data/treelist_2022a_3.csv'
tree_inventory_save_path = project_folder + 'data/fastfuels/tree_inventory_1_export.csv'


create_fastfuels(boundary, site_name, product, 
                    version, interpolation_method, 
                    curing_live_herbaceous, curing_live_woody, 
                    groups, feature_masks, remove_non_burnable, 
                    uniform_fuel_moisture_value, grid_data_zip_path, live_fuel_moisture,
                    tree_inventory_save_path, grid_data_qf_path,
                    meta = False, als = False, tree_inventory = True, tree_inventory_path = tree_inventory_path,
                    horizontal_resolution=2.0, vertical_resolution=1.0)

In [ ]:
import bp3d
import shutil
import zipfile
from scipy.io import FortranFile 
import numpy as np
import zarr
import matplotlib.pyplot as plt
import requests
from PIL import Image
from io import BytesIO
import os
import json


In [ ]:
# need to remove blackline
def raster_blackline(gui_dir, crs, array_sz):
    f = open(gui_dir + 'orig_fuels/GUIdata.json',)
    guidata = json.load(f)
    
    blackline_file = gui_dir + 'ff_blackline.geojson'
    blackline = gpd.read_file(blackline_file)
    
    # crs = 32600 + int(guidata['Location']['QF Domain']['Zone Number'])
    blackline = blackline.to_crs(crs)
    
    blackline['id'] = 1
    boundary = gpd.read_file(gui_dir + 'ff_domain.geojson')
    boundary = boundary.to_crs(blackline.crs)
    intersection = boundary.intersection(blackline) 
    
    lowerx = np.float64(guidata['Location']['QF Domain']['SW Coordinate X'])
    lowery = np.float64(guidata['Location']['QF Domain']['SW Coordinate Y'])
    
    # lowerxy = gpd.GeoDataFrame(pd.DataFrame({'id':[0]}), geometry = gpd.points_from_xy([lowerx], [lowery]), crs = crs)
    # lowerxy = lowerxy.to_crs(5070)
    # lowerx = lowerxy.geometry.x[0]
    # lowery = lowerxy.geometry.y[0]
    
    res = 2
    transform = Affine.translation(lowerx , lowery ) * Affine.scale(res, res)
    blackline_5070 = blackline.to_crs(5070)
    
    #add control line
    boundary_gdf = gpd.read_file(gui_dir + 'ff.geojson')
    boundary_gdf = boundary_gdf.to_crs(5070)
    control_line_df = pd.DataFrame({'id':['1']})
    control_line_gdf = gpd.GeoDataFrame(control_line_df, 
                                        geometry = boundary_gdf.boundary.buffer(2),
                                        crs = boundary_gdf.crs)
    control_line_gdf = control_line_gdf.to_crs(blackline_5070.crs)
    
    black_control_5070   = blackline_5070
    black_control_5070 = pd.concat([blackline_5070, control_line_gdf], ignore_index=True)
    black_control_5070 = black_control_5070.dissolve()
    
    with rasterio.open(gui_dir + 'blackline.tif', 'w+', 
               driver='GTiff',
                height=array_sz['y'],
                width=array_sz['x'],
                count=1,
                dtype=np.float64,
                crs=5070,
                transform=transform) as out:
        
        out_arr = out.read(1)
    
        # this is where we create a generator of geom, value pairs to use in rasterizing
        shapes = ((geom,value) for geom, value in zip(black_control_5070.geometry, black_control_5070.id ))
        
        blackline_rast = features.rasterize(shapes=shapes, fill=0.0001, out=out_arr, transform=out.transform, all_touched= True)
        out.write_band(1, blackline_rast)
    
        
    blackline_rast[blackline_rast==1] = 2
    blackline_rast[blackline_rast==0] = 1
    blackline_rast[blackline_rast==2] = 0
    return(blackline_rast)


def edit_dat_blackline(gui_dir, array_sz, blackline_rast, file):
    '''
    edit surface fuels for blackline
    '''
    datarray = read_dat_file(gui_dir +  file, array_sz['z'], array_sz['y'], array_sz['x'])

    blackline_surface = datarray[0,:,:] * blackline_rast 
    datarray[0,:,:] = blackline_surface

    write_np_array_to_dat(datarray, file, gui_dir)

def write_np_array_to_dat(array, dat_name, output_dir, dtype=np.float32):
    array = np.asarray(array, dtype=dtype, order="C")  # force C order
    # array = np.asarray(array, dtype=np.float32, order="C")
    with FortranFile(output_dir + dat_name, "w") as f:
        f.write_record(array)


def read_dat_file(filename, nz, ny, nx, order= "C"):
    """
    Read in a .dat file as a numpy array.

    Parameters
    ----------
    filename : Path or str
        The path to the .dat file to read.
    nz : int
        The number of cells in the z-direction.
    ny : int
        The number of cells in the y-direction.
    nx : int
        The number of cells in the x-direction.
    order : str, optional
        The order of the array. Default is "C".

    Returns
    -------
    ndarray
        A 3D numpy array representing the data in the .dat file. The array
        has dimensions (nz, ny, nx).
    """
    if isinstance(filename, str):
        filename = (filename)

    with open(filename, "rb") as fin:
        arr = (
            FortranFile(fin)
            .read_reals(dtype="float32")
            .reshape((nz, ny, nx), order=order)
        )

    return arr


In [ ]:
def prep_bp3d(dir_path, blackline_rast, zipped_dat, zipped_zarr, ensemble_name, 
                 user, password, wind_speed, wind_direction, live_fuel_moisture, fine_dead_fuel_moisture,sim_time,
              destination_name = r'fuel-array.zip'):

    # unzip and extract .dat files
    
    extract_to_folder = dir_path
    
    # Unzip the file
    with zipfile.ZipFile(zipped_dat, 'r') as zip_ref:
        print(f"Extracting all files to '{extract_to_folder}'...")
        zip_ref.extractall(extract_to_folder)
        print('Extraction complete.')

    # make the fuel-array the new zarr file that was created through fastfuels
    shutil.copy2(zipped_zarr, destination_name)
    print(f"File '{zipped_zarr}' copied successfully to '{destination_name}'")

    # create the json file that bp3d needs
    # make_json(dir_path)

    # fix the dat files with the blackline
    z = zarr.open(destination_name)

    array_sz = {}
    array_sz['x'] = z['tree']['bulkDensity'].shape[2]
    array_sz['y'] = z['tree']['bulkDensity'].shape[1]
    array_sz['z'] = z['tree']['bulkDensity'].shape[0]

    print(array_sz)
    
    edit_dat_blackline(dir_path, array_sz, blackline_rast, 'treesfueldepth.dat')
    edit_dat_blackline(dir_path, array_sz, blackline_rast, 'treesrhof.dat')
    
    
    # set up the BP3D Client
    c = bp3d.Client(user=user, password=password)
    
    # get ready to import project from GUI Folder
    qf_ensemble = c.import_project(name = ensemble_name, path = dir_path)
    

    pattern = f'{dir_path}/ignition_pattern.json'

    ign  = qf_ensemble.add_ignition(dat = f'{dir_path}/ignite.dat', pattern = pattern)


    # set the environmental variables and desired output 
    qf_ensemble.add_runs(
                    ignition=ign,
                    wind_speed=wind_speed,
                    wind_direction=wind_direction,
                    replace=True,
                    surface_moisture= fine_dead_fuel_moisture,
                    canopy_moisture= live_fuel_moisture, 
                    quicfire_version='6.1.2-dist',
                    output = {'steps_fire': 300, 'fuels_dens':True},
                    sim_time = sim_time
                    )

    
    # run ensembles
    # qf_ensemble.execute()
    return(qf_ensemble)

In [ ]:
ensemble_name_list = ['ignition 2013', 'ignition 2013 random 1', 'ignition 2013 random 2', 'ignition 2013 random 3',
                     'ignition 2022', 'ignition 2022 random 1', 'ignition 2022 random 2', 'ignition 2022 random 3'
                     ]

grid_data_zip_path = project_folder + 'data/fastfuels/trees_2022_3.zarr.zip'

file_name = ['trees_2013', 'trees_2013_1', 'trees_2013_2', 'trees_2013_3',
                     'trees_2022', 'trees_2022_1', 'trees_2022_2', 'trees_2022_3'
                     ]
zipped_zarr_list = [project_folder + 'data/fastfuels/'+ f + '.zarr.zip' for f in file_name]
zipped_dat_list = [project_folder + 'data/fastfuels/'+ f + '.qf.zip' for f in file_name]


from datetime import datetime

date = datetime.today().strftime('%Y-%m-%d %H:%M')
ensemble_name_list = [date + e for e in ensemble_name_list]

In [ ]:
ensemble_name_list = [
                     'ignition 2022a', 'ignition 2022a random 1', 'ignition 2022a random 2', 'ignition 2022a random 3'
                     ]


file_name = [
                     'trees_2022a', 'trees_2022a_1', 'trees_2022a_2', 'trees_2022a_3'
                     ]
zipped_zarr_list = [project_folder + 'data/fastfuels/'+ f + '.zarr.zip' for f in file_name]
zipped_dat_list = [project_folder + 'data/fastfuels/'+ f + '.qf.zip' for f in file_name]


from datetime import datetime

date = datetime.today().strftime('%Y-%m-%d %H:%M')
ensemble_name_list = [date + e for e in ensemble_name_list]

In [ ]:
ensemble_name_list

In [ ]:
gui_folder = '/Users/leticia/Documents/OCT_2025/Mac_GUI/projects/plum_ranch_ignition/'

In [ ]:
dir_path = '/Users/leticia/Documents/OCT_2025/Mac_GUI/projects/plum_ranch_ignition/'
# zipped_dat = grid_data_qf_path
# zipped_zarr = grid_data_zip_path
# ensemble_name = 'test of pr 2'

user = 'user'
password = 'password'

fine_dead_fuel_moisture = [0.04, 0.05, 0.12] #fm[0]
live_fuel_moisture = 0.9
wind_speed = [0/2.237, 5/2.237, 10/2.237] #wind_speed#[0]
wind_direction = [270, 225, 315]


sim_time = 80897

blackline_rast = rasterio.open(dir_path + 'blackline.tif')
blackline_rast = blackline_rast.read(1)

blackline_rast[blackline_rast==1] = 2
blackline_rast[blackline_rast==0] = 1
blackline_rast[blackline_rast==2] = 0

for i in [7]:#np.arange(len(ensemble_name_list)):
    zipped_dat = zipped_dat_list[i]
    zipped_zarr = zipped_zarr_list[i]
    ensemble_name = ensemble_name_list[i]
    
    test = prep_bp3d(dir_path, blackline_rast, zipped_dat, zipped_zarr, ensemble_name, 
                     user, password, wind_speed, wind_direction, live_fuel_moisture, fine_dead_fuel_moisture,sim_time,
                  destination_name = dir_path + r'fuel-array.zip')
    
    test.execute()